# Adım 6: Makine Öğrenmesi + MLflow
**Kişi 3 sorumluluğu** — `feature/ml-dashboard` branch

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, Imputer
from pyspark.ml.regression import (
    LinearRegression, DecisionTreeRegressor,
    RandomForestRegressor, GBTRegressor, GeneralizedLinearRegression,
)
from pyspark.ml.evaluation import RegressionEvaluator
import mlflow
import mlflow.spark
import pandas as pd
import json
import os

FEATURE_PATH = './delta_lake/features'
MODEL_PATH   = './mlflow_data/best_model'
os.makedirs('./mlflow_data', exist_ok=True)

FEATURE_COLS = [
    'temp_range', 'month', 'season_num', 'rolling_avg_7',
    'is_extreme_temp', 'prcp_category',
    'avg_wind_speed_kmh', 'avg_sea_level_pres_hpa', 'sunshine_total_min',
]
TARGET_COL = 'avg_temp_c'

def create_spark():
    return (
        SparkSession.builder
        .appName('ClimateML')
        .master('local[*]')
        .config('spark.sql.extensions',
                'io.delta.sql.DeltaSparkSessionExtension')
        .config('spark.sql.catalog.spark_catalog',
                'org.apache.spark.sql.delta.catalog.DeltaCatalog')
        .config('spark.jars.packages',
                'io.delta:delta-core_2.12:2.4.0')
        .getOrCreate()
    )

spark = create_spark()
spark.sparkContext.setLogLevel('WARN')
df = spark.read.format('delta').load(FEATURE_PATH)
df = df.dropna(subset=[TARGET_COL] + FEATURE_COLS)
print(f'[ML] Veri yuklendi: {df.count():,} kayit')
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
print(f'[ML] Egitim: {train_df.count():,}  |  Test: {test_df.count():,}')

In [ ]:
def build_pipeline(model):
    imputer = Imputer(inputCols=FEATURE_COLS,
                     outputCols=[f'{c}_imp' for c in FEATURE_COLS])
    assembler = VectorAssembler(inputCols=[f'{c}_imp' for c in FEATURE_COLS],
                                outputCol='features')
    return Pipeline(stages=[imputer, assembler, model])

def evaluate(predictions):
    ev = RegressionEvaluator(labelCol=TARGET_COL, predictionCol='prediction')
    return {
        'rmse': round(ev.setMetricName('rmse').evaluate(predictions), 4),
        'mae':  round(ev.setMetricName('mae').evaluate(predictions), 4),
        'r2':   round(ev.setMetricName('r2').evaluate(predictions), 4),
    }

def get_feature_importance(fitted_model):
    try:
        stage = fitted_model.stages[-1]
        if hasattr(stage, 'featureImportances'):
            return dict(zip(FEATURE_COLS, stage.featureImportances.toArray().tolist()))
    except Exception:
        pass
    return {}

mlflow.set_tracking_uri('./mlruns')
mlflow.set_experiment('climate-temperature-prediction')
results = []

In [ ]:
# Linear Regression
print('[ML] LinearRegression egitiliyor...')
with mlflow.start_run(run_name='LinearRegression'):
    model = LinearRegression(labelCol=TARGET_COL, featuresCol='features', maxIter=100)
    fitted = build_pipeline(model).fit(train_df)
    metrics = evaluate(fitted.transform(test_df))
    mlflow.log_param('model', 'LinearRegression')
    mlflow.log_param('maxIter', 100)
    mlflow.log_metrics(metrics)
    print(f'  RMSE={metrics["rmse"]}  MAE={metrics["mae"]}  R2={metrics["r2"]}')
    results.append({'model': 'LinearRegression', **metrics, 'feature_importance': {}, 'fitted': fitted})

# Decision Tree
print('[ML] DecisionTree egitiliyor...')
with mlflow.start_run(run_name='DecisionTree'):
    model = DecisionTreeRegressor(labelCol=TARGET_COL, featuresCol='features', maxDepth=5)
    fitted = build_pipeline(model).fit(train_df)
    metrics = evaluate(fitted.transform(test_df))
    fi = get_feature_importance(fitted)
    mlflow.log_param('model', 'DecisionTree')
    mlflow.log_param('maxDepth', 5)
    mlflow.log_metrics(metrics)
    if fi: mlflow.log_dict(fi, 'feature_importance.json')
    print(f'  RMSE={metrics["rmse"]}  MAE={metrics["mae"]}  R2={metrics["r2"]}')
    results.append({'model': 'DecisionTree', **metrics, 'feature_importance': fi, 'fitted': fitted})

## Random Forest + GBT + Generalized Linear Regression

In [ ]:
# Random Forest
print('[ML] RandomForest egitiliyor...')
with mlflow.start_run(run_name='RandomForest'):
    model = RandomForestRegressor(labelCol=TARGET_COL, featuresCol='features', numTrees=50, seed=42)
    fitted = build_pipeline(model).fit(train_df)
    metrics = evaluate(fitted.transform(test_df))
    fi = get_feature_importance(fitted)
    mlflow.log_param('model', 'RandomForest')
    mlflow.log_param('numTrees', 50)
    mlflow.log_metrics(metrics)
    if fi: mlflow.log_dict(fi, 'feature_importance.json')
    print(f'  RMSE={metrics["rmse"]}  MAE={metrics["mae"]}  R2={metrics["r2"]}')
    results.append({'model': 'RandomForest', **metrics, 'feature_importance': fi, 'fitted': fitted})

# GBT
print('[ML] GBT egitiliyor...')
with mlflow.start_run(run_name='GBT'):
    model = GBTRegressor(labelCol=TARGET_COL, featuresCol='features', maxIter=20, seed=42)
    fitted = build_pipeline(model).fit(train_df)
    metrics = evaluate(fitted.transform(test_df))
    fi = get_feature_importance(fitted)
    mlflow.log_param('model', 'GBT')
    mlflow.log_param('maxIter', 20)
    mlflow.log_metrics(metrics)
    if fi: mlflow.log_dict(fi, 'feature_importance.json')
    print(f'  RMSE={metrics["rmse"]}  MAE={metrics["mae"]}  R2={metrics["r2"]}')
    results.append({'model': 'GBT', **metrics, 'feature_importance': fi, 'fitted': fitted})

# Generalized Linear Regression
print('[ML] GeneralizedLinearReg egitiliyor...')
with mlflow.start_run(run_name='GeneralizedLinearReg'):
    model = GeneralizedLinearRegression(labelCol=TARGET_COL, featuresCol='features',
                                        family='gaussian', link='identity')
    fitted = build_pipeline(model).fit(train_df)
    metrics = evaluate(fitted.transform(test_df))
    mlflow.log_param('model', 'GeneralizedLinearReg')
    mlflow.log_metrics(metrics)
    print(f'  RMSE={metrics["rmse"]}  MAE={metrics["mae"]}  R2={metrics["r2"]}')
    results.append({'model': 'GeneralizedLinearReg', **metrics, 'feature_importance': {}, 'fitted': fitted})